# 01 — Data Collection & Alignment (Expanded)

**BBL514E Pattern Recognition — Term Project**

Bu notebook:
1. BTC (2014→) ve ETH (2017→) fiyat verilerini indirir (yfinance)
2. 4 kategori, 15 ticker makroekonomik veri indirir (2007→)
3. 1 günlük timezone lag ile tüm verileri hizalar
4. Veri kalitesi kontrollerini yapar

**Makro Kategoriler:**
- Risk Appetite: S&P 500, VIX, DXY
- Commodities: Gold, Silver, Oil WTI
- Bond Yields: US 10Y, 5Y, 3M, 30Y, 2Y
- Credit: HY Bond, IG Bond, Treasury 20Y, TIPS

**Çıktılar:**
- `data/raw/` → Ham veriler (ayrı kategorilerde)
- `data/processed/` → Hizalanmış veriler (btc_aligned.csv, eth_aligned.csv)

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

from src.utils.config import cfg
config = cfg()
print("Config loaded.")
print(f"Symbols: {list(config['data']['symbols'].keys())}")
print(f"Macro lag days: {config['data']['macro_lag_days']}")
print(f"Macro start: {config['data']['macro_start_date']}")

## 1. Fiyat Verisi Toplama (BTC: 2014→, ETH: 2017→)

Her coin kendi `start_date`'inden itibaren indiriliyor (config.yaml'da tanımlı).

In [ ]:
from src.data.price_collector import collect_all_prices

prices = collect_all_prices(save=True)

for coin, df in prices.items():
    print(f"\n{'='*60}")
    print(f"{coin.upper()} — {len(df)} rows")
    print(f"Config start_date: {config['data']['symbols'][f'{coin.upper()}-USD']['start_date']}")
    print(f"Actual date range: {df.index.min().date()} → {df.index.max().date()}")
    print(f"NaN count: {df.isna().sum().sum()}")
    print(df.describe())

In [ ]:
# Fiyat grafikleri — her coin kendi tarih aralığında
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=False)

colors = {"btc": "#F7931A", "eth": "#627EEA"}
for ax, (coin, df) in zip(axes, prices.items()):
    ax.plot(df.index, df["Close"], linewidth=0.8, color=colors.get(coin, "steelblue"))
    ax.set_title(f"{coin.upper()}/USD — Daily Close ({df.index.min().date()} → {df.index.max().date()}, {len(df)} days)", fontsize=12)
    ax.set_ylabel("Price (USD)")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/price_history.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Makroekonomik Veri Toplama (4 Kategori, 15 Ticker)

Tüm makro veriler `macro_start_date` (2007-04-11) → `end_date` (2025-12-31) arasında indiriliyor.

| Kategori | Tickers | Amaç |
|----------|---------|------|
| Risk | SP500, VIX, DXY | Piyasa risk iştahı |
| Commodities | Gold, Silver, Oil | Emtia / değer saklama |
| Yields | US 10Y/5Y/3M/30Y/2Y | Faiz ortamı |
| Credit | HYG, LQD, TLT, TIP | Kredi spreadi, enflasyon |

In [ ]:
from src.data.macro_collector import collect_all_macro

macro = collect_all_macro(save=True)

for key, df in macro.items():
    print(f"\n{'='*60}")
    if df is not None and hasattr(df, 'columns') and not df.empty:
        print(f"{key.upper()}: {len(df)} rows, {len(df.columns)} columns")
        print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
        print(f"Columns: {list(df.columns)}")
        print(df.describe())
    else:
        print(f"{key.upper()}: Not available")

In [ ]:
# Makro göstergeler — 4 kategori ayrı ayrı
fig, axes = plt.subplots(4, 4, figsize=(20, 16))

categories = {
    "risk": {"color": "#E74C3C", "title": "Risk Appetite"},
    "commodities": {"color": "#F39C12", "title": "Commodities"},
    "yields": {"color": "#3498DB", "title": "Bond Yields"},
    "credit": {"color": "#2ECC71", "title": "Credit & Inflation"},
}

for row_idx, (cat_key, cat_info) in enumerate(categories.items()):
    df = macro[cat_key]
    for col_idx in range(4):
        ax = axes[row_idx, col_idx]
        if col_idx < len(df.columns):
            col = df.columns[col_idx]
            ax.plot(df.index, df[col], linewidth=0.6, color=cat_info["color"])
            ax.set_title(f"{col}", fontsize=10)
            ax.tick_params(axis='x', labelsize=7, rotation=30)
            ax.grid(True, alpha=0.3)
        else:
            ax.set_visible(False)
    # Row label
    axes[row_idx, 0].set_ylabel(cat_info["title"], fontsize=10, fontweight="bold")

plt.suptitle("Macroeconomic Indicators (2007 → 2025, all 15 tickers)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../reports/macro_indicators.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Veri Hizalama (Alignment + Timezone Lag)

**Timezone kuralı:** NYSE kapanış 21:00 UTC, kripto günlük mum 00:00 UTC kapanır.
Bu yüzden T günü makro kapanışı ancak T+1 günü kripto sinyali için kullanılabilir.

→ Tüm makro veriler **1 gün ileri kaydırılıyor** (forward shift).
→ Sonra kripto 24/7 takvimine forward-fill ile hizalanıyor.

In [ ]:
from src.data.data_aligner import create_aligned_dataset

# Prepare macro dict for aligner (exclude fred)
macro_for_align = {
    k: v for k, v in macro.items() 
    if k != "fred" and v is not None and not v.empty
}
fred_data = macro.get("fred", None)

aligned = {}
for coin in ["btc", "eth"]:
    print(f"\n{'='*60}")
    print(f"Aligning {coin.upper()}...")
    aligned[coin] = create_aligned_dataset(
        price_df=prices[coin],
        macro_dfs=macro_for_align,
        fred_monthly=fred_data,
        coin_name=coin,
        save=True,
    )
    df = aligned[coin]
    print(f"\n  Shape: {df.shape}")
    print(f"  Date range: {df.index.min().date()} → {df.index.max().date()}")
    print(f"  NaN count: {df.isna().sum().sum()}")
    print(f"  Columns ({len(df.columns)}): {list(df.columns)}")

## 4. Veri Kalitesi Kontrolleri

Assertion-based checks — herhangi bir kontrolde sorun varsa hata verir.

In [ ]:
print("=== Data Quality Checks ===\n")

for coin in ["btc", "eth"]:
    df = aligned[coin]
    print(f"--- {coin.upper()} ---")
    
    # 1. NaN check
    nan_count = df.isna().sum().sum()
    assert nan_count == 0, f"{coin} has {nan_count} NaN values!"
    print(f"  [PASS] No NaN values")
    
    # 2. Duplicate index check
    dupes = df.index.duplicated().sum()
    assert dupes == 0, f"{coin} has {dupes} duplicate dates!"
    print(f"  [PASS] No duplicate dates")
    
    # 3. Monotonic index check
    assert df.index.is_monotonic_increasing, f"{coin} index not monotonic!"
    print(f"  [PASS] Index is monotonic increasing")
    
    # 4. Minimum row count check
    min_rows = 2500 if coin == "btc" else 2000
    assert len(df) >= min_rows, f"{coin} has only {len(df)} rows (expected >= {min_rows})"
    print(f"  [PASS] Sufficient rows: {len(df)} >= {min_rows}")
    
    # 5. Expected columns check
    expected_macro = ["SP500", "VIX", "DXY", "Gold", "Silver", "Oil_WTI",
                      "US10Y", "US5Y", "US3M", "US30Y", "US2Y",
                      "HY_Bond", "IG_Bond", "Treasury20Y", "TIPS"]
    for col in expected_macro:
        assert col in df.columns, f"{coin} missing column: {col}"
    print(f"  [PASS] All 15 macro columns present")
    
    # 6. Price sanity check
    assert (df["Close"] > 0).all(), f"{coin} has non-positive close prices!"
    print(f"  [PASS] All close prices positive")
    
    print()

In [ ]:
# Korelasyon matrisi — hizalanmış veri (sadece makro sütunları)
macro_cols = ["SP500", "VIX", "DXY", "Gold", "Silver", "Oil_WTI",
              "US10Y", "US5Y", "US3M", "US30Y", "US2Y",
              "HY_Bond", "IG_Bond", "Treasury20Y", "TIPS"]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, coin in zip(axes, ["btc", "eth"]):
    # Close + all macro columns
    cols_to_corr = ["Close"] + macro_cols
    corr = aligned[coin][cols_to_corr].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, vmin=-1, vmax=1, ax=ax, annot_kws={"size": 6},
                square=True)
    ax.set_title(f"{coin.upper()} — Close + Macro Correlation", fontsize=12)
    ax.tick_params(axis='both', labelsize=7)

plt.tight_layout()
plt.savefig("../reports/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Rapor Özet Tablosu & Timezone Lag Doğrulama

In [ ]:
# Final rapor veri özet tablosu
summary_rows = []
for coin in ["btc", "eth"]:
    df = aligned[coin]
    summary_rows.append({
        "Asset": coin.upper(),
        "Source": "yfinance",
        "Period": f"{df.index.min().date()} → {df.index.max().date()}",
        "Total Rows": len(df),
        "OHLCV Cols": 5,
        "Macro Cols": 15,
        "Total Cols": df.shape[1],
        "NaN": 0,
        "Timezone Lag": "1 day",
    })

summary = pd.DataFrame(summary_rows)
print("=== DATASET SUMMARY (for final report) ===")
print(summary.to_string(index=False))
print()

# Macro category breakdown
print("=== MACRO DATA BREAKDOWN ===")
cat_summary = []
for key, df in macro.items():
    if df is not None and hasattr(df, 'columns') and not df.empty:
        cat_summary.append({
            "Category": key.capitalize(),
            "Tickers": len(df.columns),
            "Columns": ", ".join(df.columns),
            "Raw Rows": len(df),
            "Start": df.index.min().date(),
            "End": df.index.max().date(),
        })
cat_df = pd.DataFrame(cat_summary)
print(cat_df.to_string(index=False))

## Checkpoint — Kullanıcı İncelemesi

**Bu notebook'u PyCharm'da çalıştır ve aşağıdaki noktaları kontrol et:**

### Fiyat Verileri
1. BTC grafiği 2014'ten itibaren mantıklı mı? (2017 bull run, 2018 crash, 2021 ATH, vb.)
2. ETH grafiği 2017'den itibaren doğru mu?
3. 4,111 (BTC) ve 2,967 (ETH) satır yeterli mi?

### Makro Veriler  
4. 15 makro ticker'ın zaman serileri mantıklı mı?
5. Bilinen olaylar görünüyor mu? (2008 krizi, 2020 COVID crash, 2022 rate hikes)
6. Korelasyon matrisinde sürpriz var mı?

### Teknik Kararlar
7. 1 günlük timezone lag uygulandı — onaylıyor musun?
8. Forward-fill hizalama kabul edilebilir mi?
9. FRED API key'e ihtiyaç var mı yoksa 15 yfinance ticker yeterli mi?

### Sonraki Adım
Bu fazı onaylarsan **FAZ 2: Feature Engineering**'e geçeceğiz:
- Log returns, rolling z-scores, spread variables
- Teknik indikatörler (trend, osilatör, volatilite, hacim)
- Durağanlık testleri
- Feature selection